In [ ]:
!ollama pull deepseek-r1:1.5b

In [ ]:
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

In [ ]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}
class Website:
    def __init__(self,url):
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content,'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script","style","img","input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n",strip=True)

In [ ]:
OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type":"application/json"}
# MODEL = "llama3.2"
MODEL = "deepseek-r1:1.5b"

In [ ]:
def user_prompt(website):
    prompt = f"You are looking at a website titled {website.title}"
    prompt += "\nThe contents of this website is as follows; \
    please provide a short summary of this website in markdown with the bolded headings, sub headings and also the bullet points. \
    If it includes newsor announcements, then summarize these too.\n\n"
    prompt += website.text
    return prompt

In [ ]:
webLink = Website("https://storyset.com/")

In [ ]:
message = [
    {"role":"system","content":"you are a smart assistant"},
    {"role":"user","content":user_prompt(webLink)}
]

In [ ]:
payload = {
    "model": MODEL,
    "messages": message,
    "stream":False
}

In [ ]:
response = requests.post(OLLAMA_API,json=payload,headers=HEADERS)

In [ ]:
print(response.json()['message']['content'])
display(Markdown(response.json()['message']['content']))